# SQL Master Decision Tree

**Purpose:** One unified flowchart to navigate from any SQL problem statement to the right approach. Instead of choosing between three separate strategy guides, start here — the tree will route you to the correct technique and tell you which guide to dig into for details.

**How to use this guide:** Read the problem statement, then walk down the tree answering each question. You'll land on the right pattern in under 30 seconds.

---

### Related Guides

| Guide | Use When |
|---|---|
| <a href='sql_single_table_query_strategies.html'>Single-Table Strategies</a> | One table — filter, group, rank, compare, string ops, rolling calculations |
| <a href='sql_multi_table_query_strategies.html'>Multi-Table Strategies</a> | Two+ tables — choose the right join, anti-join, self-join, CROSS JOIN |
| <a href='sql_combined_strategy_patterns.html'>Combined Strategy Patterns</a> | Join + transformation in the same query (rates, counts, ranks across tables) |
| <a href='sql_debugging_guide.html'>SQL Debugging Guide</a> | Stuck on an error? Systematic debugging for PostgreSQL |

### Decision Tree Steps

| Step | Question | Section |
|---|---|---|
| <a href='#master-tree'>Step 1</a> | How many tables? | Routes to Step 2A or 2B |
| <a href='#single-table-path'>Step 2A</a> | Single-table — what am I doing? | 14 patterns: filter → string → date → compare → aggregate → rank → roll → label → pivot → missing → dedup → delete |
| <a href='#multi-table-path'>Step 2B</a> | Multi-table — how do they relate? | 8 join types: UNION → Self → CROSS → Anti → Range → "Bought All" → LEFT → INNER |
| <a href='#combo-path'>Step 2C</a> | After the join — what transformation? | Rate, count, rank, filter groups |
| <a href='#date-fork'>Step 3</a> | Date fork — what kind of date work? | Compare rows, extract parts, calculate difference, filter range |
| <a href='#subquery-decision'>Step 4</a> | Do I need a subquery? | SELECT vs WHERE vs FROM placement |

<hr style="border: 3px solid black;">

<a id='master-tree'></a>

## Step 1: How Many Tables?

This is always the first question. Everything else follows from it.

<div class="fc">
  <div class="fc-node fc-start">READ THE PROBLEM STATEMENT<br/><br/>1. How many tables are mentioned?<br/>2. What does the expected output look like?<br/>3. Are there aggregation words (total, average, %)?<br/>4. Are there NULL / string / date clues?</div>
  <div class="fc-arrow">▼</div>
  <div class="fc-node fc-start">How many tables are involved in this problem?</div>
  <div class="fc-arrow">▼</div>
  <div class="fc-branches">
    <div class="fc-branch">
      <div class="fc-label fc-tag">ONE TABLE</div>
      <div class="fc-node fc-action"><a href="#single-table-path" data-no-arrow>Go to <strong>Step 2A: Single-Table Path →</strong></a></div>
    </div>
    <div class="fc-branch">
      <div class="fc-label fc-tag">TWO OR MORE TABLES</div>
      <div class="fc-node fc-action"><a href="#multi-table-path" data-no-arrow>Go to <strong>Step 2B: Multi-Table Path →</strong></a></div>
    </div>
  </div>
</div>

> **Watch out — "one table" can be tricky.** A table that references itself (e.g., manager_id pointing to the same Employees table) is a **Self JOIN** problem. If the problem says "compare to previous row" and there's only one table, you still go to Step 2A — use **LAG/LEAD**, not a self-join.

<hr style="border: 3px solid black;">

<a id='single-table-path'></a>

## Step 2A: Single-Table Path

You only have one table. Walk through these questions in order — **the first YES is your answer.**

> **NULL trap:** If filtering involves `!= value`, remember that `NULL != value` is NULL (not TRUE). Add `OR col IS NULL` to include NULLs. <em>Example: "Find customers NOT referred by id 2" needs `WHERE referee_id != 2 OR referee_id IS NULL`.</em>

<div class="fc">
  <div class="fc-node fc-start">SINGLE TABLE — What am I doing?</div>
  <div class="fc-arrow">▼</div>

  <div class="fc-node fc-start">Just filtering rows by conditions on columns?<br/><small>(both, either/or, not equal, greater than, odd/even)</small></div>
  <div class="fc-node fc-good">YES → <a href="sql_single_table_query_strategies.html"><strong>WHERE + ORDER BY</strong></a><code>SELECT cols FROM table
WHERE condition1 AND condition2
ORDER BY col</code>
<em>Watch for NULLs — col != X does NOT include NULLs</em></div>
  <div class="fc-arrow fc-else">if no ▼</div>

  <div class="fc-node fc-start">Does it involve string / text manipulation?<br/><small>(fix names, pattern match, valid emails, length, contains prefix)</small></div>
  <div class="fc-node fc-good">YES → <a href="sql_single_table_query_strategies.html"><strong>String functions</strong></a><code>-- Pattern matching:
WHERE col LIKE 'DIAB1%' OR col LIKE '% DIAB1%'
-- Regex (Postgres):
WHERE col ~ '^[A-Za-z][A-Za-z0-9._-]*@domain\\.com$'
-- Fix capitalization:
CONCAT(UPPER(LEFT(name,1)), LOWER(SUBSTRING(name,2)))
-- Filter by length:
WHERE LENGTH(content) > 15</code></div>
  <div class="fc-arrow fc-else">if no ▼</div>

  <div class="fc-node fc-start">Does it involve dates?<br/><small>(per month, yesterday, past 30 days, first order date)</small></div>
  <div class="fc-node fc-good">YES → <a href="#date-fork" data-no-arrow><strong>See DATE FORK (Step 3) →</strong></a> then come back here for the structural pattern</div>
  <div class="fc-arrow fc-else">if no ▼</div>

  <div class="fc-node fc-start">Am I comparing rows to each other?<br/><small>(previous, next, consecutive, yesterday vs today, swap adjacent)</small></div>
  <div class="fc-node fc-good">YES → <a href="sql_single_table_query_strategies.html"><strong>LAG() / LEAD()</strong></a> — window functions<code>SELECT *,
  LAG(col)  OVER(ORDER BY date_col) AS prev_val,
  LEAD(col) OVER(ORDER BY date_col) AS next_val
FROM table</code>
<em>For "3+ consecutive same value" → check LAG(col,1) AND LAG(col,2)</em></div>
  <div class="fc-arrow fc-else">if no ▼</div>

  <div class="fc-node fc-start">Am I collapsing rows into a summary?<br/><small>(count, sum, average, number of distinct, total per group)</small></div>
  <div class="fc-node fc-good">YES → <a href="sql_single_table_query_strategies.html"><strong>GROUP BY + aggregate function</strong></a><code>SELECT group_col, COUNT(*), SUM(val), AVG(val),
       COUNT(DISTINCT other_col)
FROM table
GROUP BY group_col</code>
<em>STRING_AGG(col, ',') to concatenate values per group</em></div>
  <div class="fc-arrow fc-else">if no ▼</div>

  <div class="fc-node fc-start">Filtering AFTER grouping?<br/><small>("at least 5 students", "appeared only once", "groups where count > N")</small></div>
  <div class="fc-node fc-good">YES → <a href="sql_single_table_query_strategies.html"><strong>GROUP BY + HAVING</strong></a><code>SELECT group_col, COUNT(*)
FROM table
GROUP BY group_col
HAVING COUNT(*) >= 5</code>
<em>"Biggest single number" → HAVING COUNT = 1, then MAX</em></div>
  <div class="fc-arrow fc-else">if no ▼</div>

  <div class="fc-node fc-start">Computing a rate, ratio, or conditional metric?<br/><small>(confirmation rate, % approved, poor query %, immediate %)</small></div>
  <div class="fc-node fc-good">YES → <a href="sql_single_table_query_strategies.html"><strong>Conditional aggregation</strong></a> — CASE inside SUM or AVG<code>SELECT group_col,
  ROUND(AVG(CASE WHEN state = 'approved'
    THEN 1.0 ELSE 0 END) * 100, 2) AS approved_pct,
  SUM(CASE WHEN state = 'approved'
    THEN amount ELSE 0 END) AS approved_total
FROM table
GROUP BY group_col</code></div>
  <div class="fc-arrow fc-else">if no ▼</div>

  <div class="fc-node fc-start">Am I ranking rows?<br/><small>("top N per group", "second highest", "nth largest", "top three salaries")</small></div>
  <div class="fc-node fc-good">YES → <a href="sql_single_table_query_strategies.html"><strong>ROW_NUMBER() / RANK() / DENSE_RANK()</strong></a><code>SELECT * FROM (
  SELECT *, DENSE_RANK()
    OVER(PARTITION BY group_col
         ORDER BY val DESC) AS rn
  FROM table
) sub WHERE rn <= 3</code>
<em>ROW_NUMBER = unique ranks; DENSE_RANK = ties share rank</em></div>
  <div class="fc-arrow fc-else">if no ▼</div>

  <div class="fc-node fc-start">Running total, cumulative sum, or rolling window?<br/><small>(running sum, moving average, 7-day window, last person under limit)</small></div>
  <div class="fc-node fc-good">YES → <a href="sql_single_table_query_strategies.html"><strong>SUM() / AVG() OVER()</strong></a> — window function<code>-- Running total:
SUM(amount) OVER(ORDER BY turn) AS running_total
-- 7-day rolling average (pre-aggregate first):
AVG(daily_total) OVER(
  ORDER BY visit_date
  ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
) AS rolling_7_avg</code></div>
  <div class="fc-arrow fc-else">if no ▼</div>

  <div class="fc-node fc-start">Labeling / categorizing / swapping?<br/><small>("if X then Y", "triangle judgement", "swap seats", salary categories)</small></div>
  <div class="fc-node fc-good">YES → <a href="sql_single_table_query_strategies.html"><strong>CASE WHEN</strong></a><code>SELECT *,
  CASE WHEN x + y > z AND x + z > y AND y + z > x
       THEN 'Yes' ELSE 'No' END AS triangle
FROM table</code>
<em>For "must include all categories even with zero count" → generate categories with VALUES or UNION, then LEFT JOIN</em></div>
  <div class="fc-arrow fc-else">if no ▼</div>

  <div class="fc-node fc-start">Pivoting rows → columns?</div>
  <div class="fc-node fc-good">YES → <a href="sql_single_table_query_strategies.html"><strong>CASE + GROUP BY (pivot)</strong></a><code>SELECT group_col,
  SUM(CASE WHEN category = 'A' THEN val ELSE 0 END) AS A,
  SUM(CASE WHEN category = 'B' THEN val ELSE 0 END) AS B
FROM table
GROUP BY group_col</code></div>
  <div class="fc-arrow fc-else">if no ▼</div>

  <div class="fc-node fc-start">Finding missing data or "NOT IN" another set?</div>
  <div class="fc-node fc-good">YES → <strong>NOT EXISTS / NOT IN / IS NULL</strong><code>SELECT * FROM table t1
WHERE NOT EXISTS (
  SELECT 1 FROM table t2
  WHERE t2.id = t1.related_id
)</code></div>
  <div class="fc-arrow fc-else">if no ▼</div>

  <div class="fc-node fc-start">Removing or identifying duplicates?</div>
  <div class="fc-node fc-good">YES → <strong>ROW_NUMBER() to deduplicate</strong><code>-- Keep one row per key:
SELECT * FROM (
  SELECT *, ROW_NUMBER()
    OVER(PARTITION BY key_col ORDER BY id ASC) AS rn
  FROM table
) sub WHERE rn = 1</code></div>
  <div class="fc-arrow fc-else">if no ▼</div>

  <div class="fc-node fc-warn">DELETE statement required?<br/><small>(delete duplicates, keep smallest id)</small></div>
  <div class="fc-node fc-good">YES → <strong>DELETE with self-reference</strong><code>DELETE FROM Person p1
USING Person p2
WHERE p1.email = p2.email
  AND p1.id > p2.id</code></div>
</div>

> **After choosing your pattern →** check <a href="#subquery-decision"><strong>Step 4: Do I Need a Subquery?</strong></a> if your formula mixes aggregation levels (e.g., "percentage of total", "above the average").

<hr style="border: 3px solid black;">

<a id='multi-table-path'></a>

## Step 2B: Multi-Table Path

Two or more tables. Walk through these questions in order — **the first YES is your JOIN type.**

<div class="fc">
  <div class="fc-node fc-start">MULTIPLE TABLES — How do they relate?</div>
  <div class="fc-arrow">▼</div>

  <div class="fc-node fc-start">Two separate questions combined into one result?<br/><small>(e.g., "user who rated most movies" AND "movie with highest avg rating")</small></div>
  <div class="fc-node fc-good">YES → <a href="sql_multi_table_query_strategies.html"><strong>UNION ALL (separate queries)</strong></a><br/>Write each query independently, stack with UNION ALL<code>(SELECT name AS results FROM ... ORDER BY ... LIMIT 1)
UNION ALL
(SELECT title AS results FROM ... ORDER BY ... LIMIT 1)</code></div>
  <div class="fc-arrow fc-else">if no ▼</div>

  <div class="fc-node fc-start">Same structure / similar data to combine?<br/><small>("combine", "merge", "both requesters and accepters")</small></div>
  <div class="fc-node fc-good">YES → <a href="sql_multi_table_query_strategies.html"><strong>UNION ALL (or UNION)</strong></a><br/>Stack rows from both sources<code>SELECT requester_id AS id FROM RequestAccepted
UNION ALL
SELECT accepter_id AS id FROM RequestAccepted</code>
<em>UNION ALL keeps dupes (faster); UNION removes them</em></div>
  <div class="fc-arrow fc-else">if no ▼</div>

  <div class="fc-node fc-start">Table references itself?<br/><small>("manager", "parent", "reports to", "same table twice")</small></div>
  <div class="fc-node fc-good">YES → <a href="sql_multi_table_query_strategies.html"><strong>Self JOIN</strong></a><code>SELECT a.name AS employee,
       b.name AS manager
FROM employees a
JOIN employees b ON a.manager_id = b.id</code>
<em>"Manager left" = LEFT JOIN self, WHERE manager IS NULL</em></div>
  <div class="fc-arrow fc-else">if no ▼</div>

  <div class="fc-node fc-start">Need ALL combinations?<br/><small>("every student with every subject", generate all pairs)</small></div>
  <div class="fc-node fc-good">YES → <a href="sql_multi_table_query_strategies.html"><strong>CROSS JOIN</strong></a><code>SELECT s.student_name, sub.subject_name
FROM Students s
CROSS JOIN Subjects sub</code>
<em>Often followed by LEFT JOIN to count actuals vs all possibilities</em></div>
  <div class="fc-arrow fc-else">if no ▼</div>

  <div class="fc-node fc-start">Need to find what's MISSING?<br/><small>("never", "didn't", "no match", "visited but no transaction")</small></div>
  <div class="fc-node fc-good">YES → <a href="sql_multi_table_query_strategies.html"><strong>LEFT JOIN + WHERE IS NULL</strong></a><br/>(anti-join pattern)<code>SELECT a.*
FROM customers a
LEFT JOIN orders b ON a.id = b.customer_id
WHERE b.customer_id IS NULL</code></div>
  <div class="fc-arrow fc-else">if no ▼</div>

  <div class="fc-node fc-start">Joining on a date range or non-equality condition?<br/><small>("price valid between start and end date", "within N days")</small></div>
  <div class="fc-node fc-good">YES → <a href="sql_multi_table_query_strategies.html"><strong>Range JOIN (non-equi)</strong></a><code>SELECT a.*, b.price
FROM UnitsSold a
JOIN Prices b
  ON a.product_id = b.product_id
  AND a.purchase_date
    BETWEEN b.start_date AND b.end_date</code></div>
  <div class="fc-arrow fc-else">if no ▼</div>

  <div class="fc-node fc-start">"Bought ALL products" / must match every item in a set?<br/><small>("customers who bought all products", relational division)</small></div>
  <div class="fc-node fc-good">YES → <a href="sql_multi_table_query_strategies.html"><strong>GROUP BY + HAVING COUNT(DISTINCT) = subquery</strong></a><code>SELECT customer_id
FROM Customer
GROUP BY customer_id
HAVING COUNT(DISTINCT product_key) =
  (SELECT COUNT(*) FROM Product)</code></div>
  <div class="fc-arrow fc-else">if no ▼</div>

  <div class="fc-node fc-start">Must keep ALL rows from one table even if no match?<br/><small>("show null if no match", "include users with zero orders")</small></div>
  <div class="fc-node fc-good">YES → <a href="sql_multi_table_query_strategies.html"><strong>LEFT JOIN</strong></a><code>SELECT a.*, COALESCE(b.val, 0) AS val
FROM main_table a
LEFT JOIN detail_table b ON a.id = b.fk</code></div>
  <div class="fc-arrow fc-else">if no ▼</div>

  <div class="fc-node fc-good">Matching rows only → <a href="sql_multi_table_query_strategies.html"><strong>INNER JOIN</strong></a><code>SELECT a.*, b.col
FROM table_a a
JOIN table_b b ON a.id = b.fk</code></div>
</div>

> **After choosing your JOIN →** Do you also need a transformation (aggregate, rank, rate)? Go to <a href="#combo-path" data-no-arrow><strong>Step 2C: After the JOIN →</strong></a>

<hr style="border: 3px solid black;">

<a id='combo-path'></a>

## Step 2C: After the JOIN — Pick the Transformation

You've picked your join from <a href='#multi-table-path'>Step 2B</a>. Now treat the joined result as a single table and pick the transformation.

<div class="fc">
  <div class="fc-node fc-start">You have joined data. What transformation is needed?</div>
  <div class="fc-arrow">▼</div>
  <div class="fc-branches fc-four">
    <div class="fc-branch">
      <div class="fc-node fc-action"><a href="sql_combined_strategy_patterns.html"><strong>Rate / Ratio</strong><br/>Conditional metric</a><code>SUM(CASE WHEN cond
  THEN 1.0 ELSE 0
END) / COUNT(*)</code>
<em>Confirmation rate, approval %</em></div>
    </div>
    <div class="fc-branch">
      <div class="fc-node fc-action"><a href="sql_combined_strategy_patterns.html"><strong>Count / Sum</strong><br/>per entity (incl. zeros)</a><code>SELECT a.name,
  COUNT(b.id)
FROM a LEFT JOIN b
  ON a.id = b.fk
GROUP BY a.name</code>
<em>Visits without transactions, exam attendance</em></div>
    </div>
    <div class="fc-branch">
      <div class="fc-node fc-action"><a href="sql_combined_strategy_patterns.html"><strong>Rank across</strong><br/>joined data</a><code>DENSE_RANK()
OVER(PARTITION BY dept
     ORDER BY sal DESC)</code>
<em>Top 3 salaries per department</em></div>
    </div>
    <div class="fc-branch">
      <div class="fc-node fc-action"><a href="sql_combined_strategy_patterns.html"><strong>Filter groups</strong><br/>by threshold</a><code>GROUP BY col
HAVING COUNT(*) >= n</code>
<em>Managers with 5+ reports</em></div>
    </div>
  </div>
</div>

All combo patterns above are covered in the <a href="sql_combined_strategy_patterns.html"><strong>Combined Strategy Patterns Guide</strong></a>.

> **Does your transformation also need a subquery?** Go to <a href="#subquery-decision" data-no-arrow><strong>Step 4: Do I Need a Subquery? →</strong></a>

<hr style="border: 3px solid black;">

<a id='date-fork'></a>

## Step 3: Date Fork

If your problem involves dates, this fork tells you which date tool to use. After choosing, **go back to the main tree** (<a href="#single-table-path">Step 2A</a> or <a href="#multi-table-path">Step 2B</a>) for the structural pattern.

<div class="fc">
  <div class="fc-node fc-start">What kind of date work?</div>
  <div class="fc-arrow">▼</div>
  <div class="fc-branches fc-four">
    <div class="fc-branch">
      <div class="fc-label fc-tag">Compare rows</div>
      <div class="fc-node fc-action">(yesterday, next day, consecutive)</div>
      <div class="fc-arrow">▼</div>
      <div class="fc-node fc-good">LAG / LEAD<br/>+ date math<code>LAG(date_col) OVER(
  ORDER BY date_col
) AS prev_date
-- check: date - prev = 1</code></div>
    </div>
    <div class="fc-branch">
      <div class="fc-label fc-tag">Extract parts</div>
      <div class="fc-node fc-action">(per month, per year, monthly)</div>
      <div class="fc-arrow">▼</div>
      <div class="fc-node fc-good">EXTRACT /<br/>TO_CHAR<br/>+ GROUP BY<code>TO_CHAR(date_col, 'YYYY-MM')
-- or --
EXTRACT(MONTH FROM date_col)</code></div>
    </div>
    <div class="fc-branch">
      <div class="fc-label fc-tag">Calculate diff</div>
      <div class="fc-node fc-action">(days between, gaps, age)</div>
      <div class="fc-arrow">▼</div>
      <div class="fc-node fc-good">Date subtraction<code>-- Postgres:
end_date - start_date
-- days as integer
-- MySQL:
DATEDIFF(end, start)</code></div>
    </div>
    <div class="fc-branch">
      <div class="fc-label fc-tag">Filter range</div>
      <div class="fc-node fc-action">(past 30 days, in February, before date)</div>
      <div class="fc-arrow">▼</div>
      <div class="fc-node fc-good">WHERE +<br/>date condition<code>WHERE date_col BETWEEN
  '2019-06-28' AND '2019-07-27'
-- or --
WHERE date_col < '2019-08-16'</code></div>
    </div>
  </div>
</div>

> **Key insight:** Date problems almost always combine with another pattern. The date fork tells you *which date tool* you need, then you flow back into the main tree (<a href="#single-table-path">Step 2A</a>) for the structural pattern (GROUP BY, LAG, RANK, etc.).

<hr style="border: 3px solid black;">

<a id='subquery-decision'></a>

## Step 4: Do I Need a Subquery?

After identifying your main pattern, check if a subquery is needed.

<div class="fc">
  <div class="fc-node fc-start">Does my calculation need a value from a DIFFERENT aggregation level or a different table I'm NOT joining to?<br/><br/>Examples:<br/>• "percentage of total" (group count / global count)<br/>• "above average" (row value vs. table average)<br/>• "pre-aggregate then join" (summary + detail)</div>
  <div class="fc-arrow">▼</div>

  <div class="fc-node fc-start">Do I need a different aggregation level at all?</div>
  <div class="fc-node fc-good">NO → <strong>No subquery needed — done!</strong></div>
  <div class="fc-arrow fc-else">if yes ▼</div>

  <div class="fc-node fc-start">Can I use a window function instead?<br/>(AVG() OVER(), SUM() OVER())</div>
  <div class="fc-node fc-good">YES → <strong>Window function is cleaner — use that</strong><code>AVG(val) OVER() AS overall_avg
-- adds a column without collapsing rows</code></div>
  <div class="fc-arrow fc-else">if no ▼</div>

  <div class="fc-node fc-warn">YES — YOU NEED A SUBQUERY. Where does it go?</div>
  <div class="fc-arrow">▼</div>
  <div class="fc-branches fc-three">
    <div class="fc-branch">
      <div class="fc-label fc-tag">In SELECT</div>
      <div class="fc-node fc-action">"Show a global value alongside each row"<br/><br/>Signal: percentage of total, ratio<code>SELECT col,
  COUNT(*) * 100.0 /
  (SELECT COUNT(*)
   FROM table) AS pct
FROM table
GROUP BY col</code></div>
    </div>
    <div class="fc-branch">
      <div class="fc-label fc-tag">In WHERE</div>
      <div class="fc-node fc-action">"Filter rows using a computed threshold"<br/><br/>Signal: above average, top performers<code>SELECT *
FROM table
WHERE val > (
  SELECT AVG(val)
  FROM table
)</code></div>
    </div>
    <div class="fc-branch">
      <div class="fc-label fc-tag">In FROM</div>
      <div class="fc-node fc-action">"Pre-aggregate, then query the result"<br/><br/>Signal: rank groups, filter summaries<code>SELECT * FROM (
  SELECT col, COUNT(*) AS cnt
  FROM table
  GROUP BY col
) sub
WHERE cnt > 5</code></div>
    </div>
  </div>
</div>

For detailed examples of each subquery placement, see:
- <a href="sql_subqueries_select.html"><strong>Subqueries in SELECT</strong></a> — percentage of total, ratio calculations
- <a href="sql_subqueries_where.html"><strong>Subqueries in WHERE</strong></a> — filtering by computed thresholds
- <a href="sql_subqueries_from.html"><strong>Subqueries in FROM</strong></a> — pre-aggregating before further logic


<hr style="border: 3px solid black;">

<a id='quick-reference'></a>

## Quick Reference: Signal Words → Pattern

When you're stuck, scan the problem for these words:

### Single-Table Patterns (<a href="sql_single_table_query_strategies.html">Single-Table Guide</a>)

| Signal Words in Problem | Pattern | Example Problem |
|---|---|---|
| "find rows where", "both", "either/or", "not equal" | <a href="#single-table-path">WHERE + ORDER BY</a> | Recyclable and Low Fat Products |
| "not referred by" + possible NULLs | WHERE + **IS NULL handling** | Find Customer Referee |
| "odd-numbered", "MOD", even/odd filtering | WHERE + MOD() or % | Not Boring Movies |
| "fix names", "uppercase", "lowercase" | **String: UPPER / LOWER / CONCAT** | Fix Names in a Table |
| "starts with prefix", "contains", "pattern" | **String: LIKE / REGEXP** | Patients With a Condition |
| "valid email", "matches format" | **String: ~ (regex)** | Find Users With Valid E-Mails |
| "length", "number of characters" | **String: LENGTH()** | Invalid Tweets |
| "previous", "next", "yesterday", "consecutive" | <a href="#single-table-path">LAG / LEAD</a> | Rising Temperature, Consecutive Numbers |
| "swap adjacent", "exchange pairs" | CASE WHEN + MOD + LEAD/LAG | Exchange Seats |
| "total", "average", "count per", "sum of" | <a href="#single-table-path">GROUP BY + aggregate</a> | Followers Count, Unique Subjects |
| "number of distinct", "count unique" | GROUP BY + COUNT(DISTINCT) | Unique Subjects per Teacher |
| "concatenate per group", "list products" | GROUP BY + STRING_AGG() | Group Sold Products By Date |
| "at least N", "appeared only once" | <a href="#single-table-path">GROUP BY + HAVING</a> | Classes With 5+ Students, Biggest Single Number |
| "confirmation rate", "% approved", "poor query %" | <a href="#single-table-path">Conditional aggregation</a> (CASE in SUM) | Monthly Transactions, Queries Quality |
| "immediate %", "fraction that..." | AVG(CASE WHEN ... THEN 1.0 ELSE 0 END) | Immediate Food Delivery II |
| "top N", "rank", "second highest", "nth largest" | <a href="#single-table-path">ROW_NUMBER / RANK / DENSE_RANK</a> | Second Highest Salary |
| "first year", "earliest occurrence" | RANK + filter rn = 1 | Product Sales Analysis III |
| "running total", "cumulative", "weight limit" | <a href="#single-table-path">SUM() OVER()</a> | Last Person to Fit in the Bus |
| "moving average", "7-day window", "rolling" | SUM/AVG OVER(ROWS BETWEEN) | Restaurant Growth |
| "label", "categorize", "if...then", "triangle" | <a href="#single-table-path">CASE WHEN</a> | Triangle Judgement, Count Salary Categories |
| "only department" OR "primary flag" (conditional logic) | CASE WHEN or UNION | Primary Department for Each Employee |
| "pivot", "rows to columns" | CASE + GROUP BY | Monthly Transactions I |
| "missing", "NOT IN", "doesn't exist" | NOT EXISTS / IS NULL | Employees Whose Manager Left |
| "duplicates", "unique", "first occurrence" | ROW_NUMBER / DISTINCT | Product Price at Given Date |
| "**delete** duplicates", "keep smallest id" | DELETE + self-reference | Delete Duplicate Emails |

### Multi-Table Patterns (<a href="sql_multi_table_query_strategies.html">Multi-Table Guide</a>)

| Signal Words in Problem | Pattern | Example Problem |
|---|---|---|
| Two independent questions → one result | <a href="#multi-table-path">UNION ALL (separate queries)</a> | Movie Rating |
| "both sides" of a relationship, "requesters AND accepters" | <a href="#multi-table-path">UNION ALL (reshape)</a> | Friend Requests: Most Friends |
| "manager", "reports to", "parent", self-referencing FK | <a href="#multi-table-path">Self JOIN</a> | Managers with 5+ Reports |
| "every X with every Y", "all combinations" | <a href="#multi-table-path">CROSS JOIN</a> | Students and Examinations |
| "never", "didn't", "no match", "without", "not make" | <a href="#multi-table-path">LEFT JOIN + IS NULL</a> | Visited but No Transactions |
| "price valid between dates", range condition | <a href="#multi-table-path">Range JOIN (BETWEEN)</a> | Average Selling Price |
| "bought ALL products", "matches every item" | <a href="#multi-table-path">HAVING COUNT(DISTINCT) = subquery</a> | Customers Who Bought All Products |
| "show null if no match", "even if no bonus" | <a href="#multi-table-path">LEFT JOIN</a> | Employee Bonus, Replace Employee ID |
| "look up name", "get details from", "enrich" | <a href="#multi-table-path">INNER JOIN</a> | Product Sales Analysis I |

### Combined Patterns (<a href="sql_combined_strategy_patterns.html">Combined Guide</a>)

| Signal Words in Problem | Pattern | Example Problem |
|---|---|---|
| "rate per entity including zeros" | LEFT JOIN + CASE + GROUP BY | Confirmation Rate |
| "count per entity including zeros" | LEFT JOIN + COUNT + GROUP BY | Students and Examinations |
| "top N per group" across tables | JOIN + DENSE_RANK | Department Top Three Salaries |
| "percentage of total" from another table | Subquery in SELECT (denominator) | Percentage of Users in Contest |
| "weighted average" with range join | Range JOIN + SUM(a*b)/SUM(b) | Average Selling Price |

<hr style="border: 3px solid black;">

<a id='full-flow'></a>

## The Complete Flow — All Steps on One Diagram

<div class="fc">
  <div class="fc-node fc-start">READ THE PROBLEM</div>
  <div class="fc-arrow">▼</div>
  <div class="fc-node fc-start"><a href="#master-tree">Step 1: How many tables?</a></div>
  <div class="fc-arrow">▼</div>
  <div class="fc-branches">
    <div class="fc-branch">
      <div class="fc-label fc-tag">ONE TABLE</div>
      <div class="fc-arrow">▼</div>
      <div class="fc-node fc-action"><a href="#single-table-path" data-no-arrow>Walk <strong>Step 2A →</strong></a><br/>(filter, string, date, compare, aggregate, rank, roll, label, pivot, missing, dedup, delete)</div>
    </div>
    <div class="fc-branch">
      <div class="fc-label fc-tag">2+ TABLES</div>
      <div class="fc-arrow">▼</div>
      <div class="fc-node fc-action"><a href="#multi-table-path" data-no-arrow>Walk <strong>Step 2B →</strong></a><br/>(UNION, self, CROSS, anti, range, "all", LEFT, INNER)</div>
      <div class="fc-arrow">▼</div>
      <div class="fc-node fc-start">Need transformation after the join?</div>
      <div class="fc-arrow">▼</div>
      <div class="fc-branches">
        <div class="fc-branch">
          <div class="fc-label fc-no">NO</div>
          <div class="fc-arrow">▼</div>
          <div class="fc-node fc-good">Done! JOIN alone answers it</div>
        </div>
        <div class="fc-branch">
          <div class="fc-label fc-yes">YES</div>
          <div class="fc-arrow">▼</div>
          <div class="fc-node fc-action"><a href="#combo-path" data-no-arrow>Walk <strong>Step 2C →</strong></a><br/>(rate, count, rank, filter)</div>
        </div>
      </div>
    </div>
  </div>
  <div class="fc-arrow">▼</div>
  <div class="fc-node fc-start">Does any part of my formula need a value from a DIFFERENT aggregation level?</div>
  <div class="fc-arrow">▼</div>
  <div class="fc-branches">
    <div class="fc-branch">
      <div class="fc-label fc-no">NO</div>
      <div class="fc-node fc-good">DONE! Write your query</div>
    </div>
    <div class="fc-branch">
      <div class="fc-label fc-yes">YES</div>
      <div class="fc-node fc-action"><a href="#subquery-decision" data-no-arrow>Walk <strong>Step 4 →</strong></a><br/>(SELECT / WHERE / FROM)</div>
    </div>
  </div>
</div>

<hr style="border: 3px solid black;">

<a id='worked-example'></a>

## Worked Examples: Walking the Tree

### Example 1: Percentage of Users Registered per Contest

**Problem:** Find the percentage of total users registered in each contest, rounded to two decimal places.

**Tables:** Users (user_id, user_name) and Register (contest_id, user_id)

**Walking the tree:**

**<a href="#master-tree">Step 1</a> — How many tables?** Two (Users and Register) → <a href="#multi-table-path">Go to Step 2B</a>

**<a href="#multi-table-path">Step 2B</a> — Pick the JOIN:** The output only needs `contest_id` and `percentage` — I don't need columns from Users, just a count. → No JOIN needed, but I need data from another table.

**<a href="#combo-path">Step 2C</a> — Transformation:** GROUP BY `contest_id` + COUNT per group → GROUP BY + aggregate

**<a href="#subquery-decision">Step 4</a> — Subquery?** Formula is `group_count / total_count * 100`. Total comes from a different table at different grain. → **Subquery in SELECT** (denominator).

```sql
SELECT r.contest_id,
       ROUND(COUNT(DISTINCT r.user_id)::numeric
             / (SELECT COUNT(*) FROM Users) * 100, 2) AS percentage
FROM Register r
GROUP BY r.contest_id
ORDER BY percentage DESC, r.contest_id ASC;
```

**Path:** Step 1 (2 tables) → 2B (skip join) → 2C (GROUP BY) → 4 (subquery in SELECT)

---

### Example 2: Restaurant 7-Day Moving Average

**Problem:** Compute the moving average of customer payments over a 7-day window (current day + 6 days before).

**Table:** Customer (customer_id, name, visited_on, amount) — **one table**

**Walking the tree:**

**<a href="#master-tree">Step 1</a> — How many tables?** One → <a href="#single-table-path">Go to Step 2A</a>

**<a href="#single-table-path">Step 2A</a> — Walk the checklist:**
- Just filtering? No.
- String manipulation? No.
- **Involves dates?** Yes → <a href="#date-fork">Step 3 Date Fork</a>: What kind? Compare rows over time + range calculation → need **date filtering** (ROWS BETWEEN) + back to 2A.
- Back in 2A: Running total / rolling window? **YES** → SUM/AVG OVER(ROWS BETWEEN).

**Key insight:** Multiple customers per day means we must pre-aggregate daily totals first, then apply the window.

```sql
WITH daily AS (
  SELECT visited_on, SUM(amount) AS day_total
  FROM Customer
  GROUP BY visited_on
)
SELECT visited_on,
       SUM(day_total) OVER(ORDER BY visited_on
         ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS amount,
       ROUND(AVG(day_total) OVER(ORDER BY visited_on
         ROWS BETWEEN 6 PRECEDING AND CURRENT ROW), 2) AS average_amount
FROM daily
ORDER BY visited_on
OFFSET 6;
```

**Path:** Step 1 (1 table) → 2A (dates → Step 3 → back to 2A → rolling window)

---

### Example 3: Students and Examinations (CROSS JOIN + LEFT JOIN + COUNT)

**Problem:** Find how many times each student attended each exam. Must show 0 for exams not attended.

**Tables:** Students, Subjects, Examinations — **three tables**

**Walking the tree:**

**<a href="#master-tree">Step 1</a>** — Three tables → <a href="#multi-table-path">Step 2B</a>

**<a href="#multi-table-path">Step 2B</a>** — I need every student paired with every subject (including zeros). → **CROSS JOIN** (Students x Subjects) to generate all combos, then LEFT JOIN Examinations to count actuals.

**<a href="#combo-path">Step 2C</a>** — Transformation: COUNT per student-subject pair → **GROUP BY + COUNT**

```sql
SELECT s.student_id, s.student_name, sub.subject_name,
       COUNT(e.student_id) AS attended_exams
FROM Students s
CROSS JOIN Subjects sub
LEFT JOIN Examinations e
  ON s.student_id = e.student_id
  AND sub.subject_name = e.subject_name
GROUP BY s.student_id, s.student_name, sub.subject_name
ORDER BY s.student_id, sub.subject_name;
```

**Path:** Step 1 (3 tables) → 2B (CROSS JOIN + LEFT JOIN) → 2C (COUNT per group)

<hr style="border: 3px solid black;">

<a id='final-takeaway'></a>

## Final Takeaway

Every SQL problem breaks down into the same four questions:

1. **How many tables?** → Determines if you need a join
2. **What join?** → Match the relationship pattern to the right join type
3. **What transformation?** → Filter, aggregate, rank, compare, or label
4. **Different grain?** → If yes, you need a subquery (and now you know where it goes)

Practice walking the tree on every problem before writing code. After a few dozen problems, the routing becomes automatic.